## 1. Setup Environment

Check if running on Colab and setup accordingly

In [ ]:
import os
import sys

# Check if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
    from google.colab import drive
    # Uncomment to mount Google Drive
    # drive.mount('/content/drive')
else:
    print("Running locally")

# Check GPU availability
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 2. Install Dependencies (Colab only)

Skip this cell if running locally with existing environment

In [ ]:
if IN_COLAB:
    !pip install -q einops
    print("Dependencies installed!")
else:
    print("Skipping - using local environment")

## 3. Upload/Setup Project Files

**For Colab:** Upload your project as a zip file or clone from GitHub

**For Local:** This will use your current directory

In [ ]:
# Upload/Setup Project Files – Download directly from Google Drive folder by ID (no manual upload)
import os, shutil
from google.colab import auth
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

IN_COLAB = 'google.colab' in globals() or 'google.colab' in __import__('sys').modules

if IN_COLAB:
    # Install PyDrive if not present
    try:
        import pydrive
    except ImportError:
        !pip install -q pydrive
        import pydrive
    
    # Drive folder ID from your link
    FOLDER_ID = "1fbaCU7nlEYu5iDuwvjPw4E3NZGaVIbnO"
    PROJECT_ROOT = "/content/pathformer-main"
    TMP_DIR = "/content/tmp_download"
    
    def ensure_clean_dir(path):
        if os.path.exists(path):
            shutil.rmtree(path)
        os.makedirs(path, exist_ok=True)
    
    # Authenticate to Google
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.CommandLineAuth()
    drive = GoogleDrive(gauth)
    
    def list_folder_files(folder_id):
        query = f"'{folder_id}' in parents and trashed=false"
        return drive.ListFile({'q': query}).GetList()
    
    def download_folder_recursively(folder_id, target_dir):
        os.makedirs(target_dir, exist_ok=True)
        items = list_folder_files(folder_id)
        for item in items:
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                subdir = os.path.join(target_dir, item['title'])
                download_folder_recursively(item['id'], subdir)
            else:
                # Skip Google-native docs (Docs/Sheets)
                if item['mimeType'].startswith('application/vnd.google-apps'):
                    print(f"Skipping Google-native file: {item['title']} ({item['mimeType']})")
                    continue
                out_path = os.path.join(target_dir, item['title'])
                print(f"Downloading: {item['title']} -> {out_path}")
                item.GetContentFile(out_path)
    
    print("Downloading Drive folder to /content/tmp_download...")
    ensure_clean_dir(TMP_DIR)
    download_folder_recursively(FOLDER_ID, TMP_DIR)
    
    print("Organizing files into /content/pathformer-main ...")
    ensure_clean_dir(PROJECT_ROOT)
    for item in os.listdir(TMP_DIR):
        src = os.path.join(TMP_DIR, item)
        dst = os.path.join(PROJECT_ROOT, item)
        shutil.move(src, dst)
    
    print("\nProject files:")
    !ls -la /content/pathformer-main
    %cd /content/pathformer-main
else:
    # Local environment: keep current directory
    project_dir = os.getcwd()
    print(f"Using project directory: {project_dir}")
    
    print("\nProject files:")
    !ls -la

## 4. Import Libraries and Set Parameters

In [ ]:
import torch
import numpy as np
import random
from exp.exp_main import Exp_Main
import argparse
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed
fix_seed = 1024
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

print("Libraries imported successfully!")

## 5. Configure Training Parameters

Modify these parameters as needed

In [ ]:
class Args:
    # Training settings
    is_training = 1
    model = 'PathFormer'
    model_id = 'electricity_96_96'
    
    # Data settings
    data = 'custom'
    root_path = './dataset/electricity'
    data_path = 'electricity.csv'
    features = 'M'  # M: multivariate, S: univariate
    target = 'OT'
    freq = 'h'
    checkpoints = './checkpoints/'
    
    # Forecasting task
    seq_len = 96
    pred_len = 96
    individual = False
    
    # Model parameters
    d_model = 16
    d_ff = 128
    num_nodes = 321
    layer_nums = 3
    k = 2
    num_experts_list = [4, 4, 4]
    patch_size_list = [[16, 12, 8, 32], [12, 8, 6, 4], [8, 6, 4, 2]]
    do_predict = True
    revin = 1
    drop = 0.1
    embed = 'timeF'
    residual_connection = 1
    metric = 'mae'
    batch_norm = 0
    
    # Training optimization
    num_workers = 10
    itr = 1
    train_epochs = 50
    batch_size = 16  # Increase if you have enough GPU memory
    patience = 10
    learning_rate = 0.001
    lradj = 'TST'
    use_amp = False
    pct_start = 0.4
    
    # GPU settings - IMPORTANT!
    use_gpu = True if torch.cuda.is_available() else False
    gpu = 0
    use_multi_gpu = False
    devices = '0'
    test_flop = False

args = Args()

print("Configuration:")
print(f"  GPU Enabled: {args.use_gpu}")
print(f"  Training Epochs: {args.train_epochs}")
print(f"  Batch Size: {args.batch_size}")
print(f"  Sequence Length: {args.seq_len}")
print(f"  Prediction Length: {args.pred_len}")

## 6. Initialize Experiment

In [ ]:
# Create experiment setting string
setting = '{}_{}_ft{}_sl{}_pl{}_{}'.format(
    args.model_id,
    args.model,
    args.data_path[:-4],
    args.features,
    args.seq_len,
    args.pred_len,
    0  # iteration number
)

print(f"Experiment: {setting}")
print(f"Using device: {'GPU' if args.use_gpu else 'CPU'}")

# Initialize experiment
exp = Exp_Main(args)
print("\nExperiment initialized successfully!")

## 7. Train Model

This will take some time depending on your settings

In [ ]:
print('='*80)
print(f'START TRAINING: {setting}')
print('='*80)

start_time = time.time()
exp.train(setting)
training_time = time.time() - start_time

print(f'\nTraining completed in {training_time/60:.2f} minutes!')

## 8. Test Model

In [ ]:
print('='*80)
print(f'TESTING: {setting}')
print('='*80)

test_start = time.time()
exp.test(setting)
test_time = time.time() - test_start

print(f'\nTesting completed in {test_time:.2f} seconds!')

## 9. Generate Predictions (Optional)

In [ ]:
if args.do_predict:
    print('='*80)
    print(f'PREDICTING: {setting}')
    print('='*80)
    
    exp.predict(setting, load=True)
    print('\nPredictions completed!')

## 10. Visualize Results

Load and visualize the predictions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
import os

# Result folder
result_folder = f'./test_results/{setting}/'

# Load saved arrays
if os.path.exists(result_folder + 'predictions.npy'):
    predictions = np.load(result_folder + 'predictions.npy')
    ground_truth = np.load(result_folder + 'ground_truth.npy')
    inputs = np.load(result_folder + 'inputs.npy')
    
    print(f"Data loaded:")
    print(f"  Predictions shape: {predictions.shape}")
    print(f"  Ground truth shape: {ground_truth.shape}")
    print(f"  Inputs shape: {inputs.shape}")
    
    # Display some generated plots
    print("\n" + "="*80)
    print("Generated Visualizations:")
    print("="*80)
    
    for i in range(min(3, len(predictions))):
        img_path = os.path.join(result_folder, f'prediction_sample_{i}.png')
        if os.path.exists(img_path):
            print(f"\nSample {i}:")
            display(Image(filename=img_path))
else:
    print("No saved predictions found. Make sure testing completed successfully.")

## 11. Custom Visualization

Create your own plots

In [ ]:
def plot_sample(sample_idx=0, feature_idx=-1):
    """
    Plot a specific sample and feature
    """
    if 'predictions' not in globals():
        print("Please run the previous cell to load data first.")
        return
    
    plt.figure(figsize=(15, 5))
    
    seq_len = inputs.shape[1]
    pred_len = predictions.shape[1]
    
    x_input = np.arange(seq_len)
    x_pred = np.arange(seq_len, seq_len + pred_len)
    
    # Plot
    plt.plot(x_input, inputs[sample_idx, :, feature_idx], 
             label='Input Sequence', color='blue', linewidth=2)
    plt.plot(x_pred, ground_truth[sample_idx, :, feature_idx], 
             label='Ground Truth', color='green', linewidth=2)
    plt.plot(x_pred, predictions[sample_idx, :, feature_idx], 
             label='Prediction', color='red', linewidth=2, linestyle='--')
    
    plt.axvline(x=seq_len, color='black', linestyle=':', linewidth=1.5)
    
    # Calculate error
    mae = np.mean(np.abs(predictions[sample_idx, :, feature_idx] - 
                          ground_truth[sample_idx, :, feature_idx]))
    
    plt.xlabel('Time Steps', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.title(f'Sample {sample_idx} - MAE: {mae:.4f}', fontsize=14)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot first sample
plot_sample(sample_idx=0, feature_idx=-1)

## 12. Download Results (Colab Only)

Download checkpoints and results

In [ ]:
if IN_COLAB:
    # Zip results
    !zip -r results.zip ./test_results ./checkpoints
    
    # Download
    from google.colab import files
    files.download('results.zip')
    print("Results downloaded!")
else:
    print("Results are already saved locally in:")
    print(f"  - ./test_results/{setting}/")
    print(f"  - ./checkpoints/{setting}/")